# Notebook 05: MLP Analysis (Level 4)

## Overview
Deep analysis of **MLP sub-layer contributions** to frequency-band processing
in the ISC induction circuit. While NB02 (Residual Stream) tracked full residual
representations and NB04 examined attention heads, this notebook isolates the
MLP component: its output geometry, separability, contribution to the correct
logit, share of total computation, neuron-level selectivity, and activation
sparsity patterns across frequency bands.

## Key Questions
- At which layers do MLP outputs most separate frequency bands?
- Can MLP representations alone predict frequency band? How does this compare to residual stream probing (NB02)?
- How much does each MLP layer contribute to the correct next-token logit, and does this differ by band?
- What fraction of total representation update comes from MLP vs attention per layer?
- Are there individual neurons that are selectively activated by specific frequency bands?
- Does MLP activation sparsity differ across bands, and does it scale with model size?

## Hypothesis Domain: R5 (MLP Contributions)
- **H-R5.1**: MLP separability peaks at mid-to-late layers (probe accuracy trajectory, per model)
- **H-R5.2**: MLP contribution to correct logit is higher for high-frequency tokens (Mann-Whitney, per model)
- **H-R5.3**: MLP contribution fraction increases with layer depth (trend test, per model)
- **H-R5.4**: Frequency-selective neurons exist (F-statistic > permutation null, per layer per model)
- **H-R5.5**: MLP activation sparsity is higher for high-frequency bands (Gini coefficient, per model)
- **H-R5.6**: Larger models exhibit more neuron selectivity and higher sparsity (scaling analysis)

## Notebook Structure
1. Setup & Data Loading
2. MLP Output Geometry
3. MLP Separability (Linear Probing)
4. MLP Contribution to Correct Logit
5. MLP Contribution Fraction (vs Attention)
6. Neuron-Level Analysis (Frequency Selectivity)
7. MLP Activation Sparsity
8. Cross-Model Comparison
9. Summary

## Data Sources
- Pre-extracted activations from NPZ files:
  - `mlp_out_predpos`: MLP output at prediction position, shape (N, n_layers, d_model)
  - `mlp_pre_predpos`: MLP pre-activations at prediction position, shape (N, n_layers, d_mlp)
  - `attn_out_predpos`: Attention output at prediction position, shape (N, n_layers, d_model)
  - `target_ids`: Target token IDs, shape (N,)
- NB02 probe results for comparison (from `02_probe_trajectory.csv`)

## 1. Setup & Data Loading

In [1]:
import sys
import numpy as np
import pandas as pd
from pathlib import Path
from scipy import stats

sys.path.insert(0, str(Path.cwd()))

from utils.constants import (
    MODELS,
    BANDS,
    DRAWS,
    FREQUENCY_RANK,
    MODEL_INFO,
    BAND_COLORS,
    BAND_NAMES,
    MODEL_COLORS,
    MODEL_CAPACITY,
    MODEL_D_MODEL,
    MODEL_D_MLP,
    ACTIVATIONS_DIR,
    ANALYSIS_DIR,
    VIZ_DIR,
    RANDOM_SEED,
    N_PERMUTATIONS,
    CV_FOLDS,
    get_domain_dirs,
)
from utils.data_loading import (
    load_extracted_activations,
    save_analysis,
    build_representational_df,
)
from utils.geometry import (
    compute_band_centroids,
    compute_centroid_distances,
    compute_within_band_spread,
    compute_separation_ratio,
    compute_band_geometry,
)
from utils.probing import train_probe, train_probe_trajectory
from utils.plotting import (
    setup_plotting,
    save_figure,
    plot_metric_heatmap,
    plot_boxplot_by_band,
)

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

setup_plotting()
ANALYSIS_DIR, VIZ_DIR = get_domain_dirs("mlp", "base")
from functools import partial as _partial

save_analysis = _partial(save_analysis, analysis_dir=ANALYSIS_DIR)
save_figure = _partial(save_figure, viz_dir=VIZ_DIR)

print(f"Models: {MODELS}")
print(f"Bands:  {BANDS}")
print(f"Draws:  {DRAWS}")

Models: ['pythia-70m', 'pythia-160m', 'pythia-410m', 'pythia-1b', 'pythia-1.4b']
Bands:  ['low', 'medium', 'high', 'very_high', 'control']
Draws:  ['draw_1', 'draw_2', 'draw_3']


In [2]:
# Load MLP output activations at prediction position
all_mlp_out = {}  # model -> draw -> {band: (N, n_layers, d_model)}
all_mlp_pre = {}  # model -> draw -> {band: (N, n_layers, d_mlp)}
all_attn_out = {}  # model -> draw -> {band: (N, n_layers, d_model)}
all_target_ids = {}  # model -> draw -> {band: (N,)}

for model in MODELS:
    all_mlp_out[model] = {}
    all_mlp_pre[model] = {}
    all_attn_out[model] = {}
    all_target_ids[model] = {}
    for draw in DRAWS:
        all_mlp_out[model][draw] = {}
        all_mlp_pre[model][draw] = {}
        all_attn_out[model][draw] = {}
        all_target_ids[model][draw] = {}
        for band in BANDS:
            try:
                data = load_extracted_activations(model, band, draw)
                if "mlp_out_predpos" in data:
                    all_mlp_out[model][draw][band] = data["mlp_out_predpos"]
                if "mlp_pre_predpos" in data:
                    all_mlp_pre[model][draw][band] = data["mlp_pre_predpos"]
                if "attn_out_predpos" in data:
                    all_attn_out[model][draw][band] = data["attn_out_predpos"]
                if "target_ids" in data:
                    all_target_ids[model][draw][band] = data["target_ids"]
            except FileNotFoundError:
                pass

# Report shapes
for model in MODELS:
    for draw in ["draw_1"]:
        sample_out = next(iter(all_mlp_out[model][draw].values()), None)
        sample_pre = next(iter(all_mlp_pre[model][draw].values()), None)
        sample_attn = next(iter(all_attn_out[model][draw].values()), None)
        shapes = []
        if sample_out is not None:
            shapes.append(f"mlp_out={sample_out.shape}")
        if sample_pre is not None:
            shapes.append(f"mlp_pre={sample_pre.shape}")
        if sample_attn is not None:
            shapes.append(f"attn_out={sample_attn.shape}")
        if shapes:
            print(f"{model}: {', '.join(shapes)}")
        else:
            print(f"{model}: no MLP activations found")

pythia-70m: mlp_out=(225, 6, 512), mlp_pre=(225, 6, 2048), attn_out=(225, 6, 512)
pythia-160m: mlp_out=(225, 12, 768), mlp_pre=(225, 12, 3072), attn_out=(225, 12, 768)
pythia-410m: mlp_out=(225, 24, 1024), mlp_pre=(225, 24, 4096), attn_out=(225, 24, 1024)
pythia-1b: mlp_out=(225, 16, 2048), mlp_pre=(225, 16, 8192), attn_out=(225, 16, 2048)
pythia-1.4b: mlp_out=(225, 24, 2048), mlp_pre=(225, 24, 8192), attn_out=(225, 24, 2048)


## 2. MLP Output Geometry

MLP output norms at the prediction position, per layer per band.
Centroid distances between bands in MLP output space.
This reveals how much the MLP sub-layer contributes in magnitude
and whether its outputs already cluster by frequency band.

In [3]:
norm_records = []

for model in MODELS:
    n_layers = MODEL_INFO[model]["n_layers"]
    for draw in DRAWS:
        for band in BANDS:
            mlp_out = all_mlp_out.get(model, {}).get(draw, {}).get(band)
            if mlp_out is None:
                continue

            for layer in range(n_layers):
                X = mlp_out[:, layer, :]  # (N, d_model)
                norms = np.linalg.norm(X, axis=1)

                norm_records.append(
                    {
                        "model": model,
                        "draw": draw,
                        "band": band,
                        "layer": layer,
                        "mean_norm": float(norms.mean()),
                        "std_norm": float(norms.std()),
                        "median_norm": float(np.median(norms)),
                    }
                )

df_norms = pd.DataFrame(norm_records)
save_analysis(df_norms, "05_mlp_output_norms.csv")
print(f"MLP output norm records: {len(df_norms)}")
df_norms.head()

MLP output norm records: 1230


,model,draw,band,layer,mean_norm,std_norm,median_norm
0,pythia-70m,draw_1,low,0,6.798147,0.567965,6.762117
1,pythia-70m,draw_1,low,1,5.311841,1.467718,4.780629
2,pythia-70m,draw_1,low,2,11.977400,14.144870,5.649688
3,pythia-70m,draw_1,low,3,9.011966,1.503523,8.527209
4,pythia-70m,draw_1,low,4,15.799229,9.857393,11.897347


In [4]:
# Plot MLP output norm trajectory per model, colored by band
for model in MODELS:
    model_data = df_norms[(df_norms["model"] == model) & (df_norms["draw"] == "draw_1")]
    if len(model_data) == 0:
        continue

    fig, ax = plt.subplots(figsize=(12, 5))
    for band in BANDS:
        bd = model_data[model_data["band"] == band].sort_values("layer")
        if len(bd) == 0:
            continue
        ax.plot(
            bd["layer"],
            bd["mean_norm"],
            color=BAND_COLORS.get(band, "gray"),
            label=BAND_NAMES.get(band, band),
            marker="o",
            markersize=3,
        )
    ax.set_xlabel("Layer")
    ax.set_ylabel("Mean L2 Norm (MLP Output)")
    ax.set_title(f"MLP Output Norm Trajectory \u2014 {model}")
    ax.legend(fontsize=8)
    fig.tight_layout()
    save_figure(fig, f"viz_05_01_mlp_norm_trajectory_{model}.png")

In [5]:
centroid_records = []

for model in MODELS:
    n_layers = MODEL_INFO[model]["n_layers"]
    for draw in DRAWS:
        for layer in range(n_layers):
            embs, labels = [], []
            for band in BANDS:
                mlp_out = all_mlp_out.get(model, {}).get(draw, {}).get(band)
                if mlp_out is None:
                    continue
                embs.append(mlp_out[:, layer, :])
                labels.extend([band] * mlp_out.shape[0])

            if len(embs) < 2:
                continue

            X = np.vstack(embs)
            y = np.array(labels)

            geom = compute_band_geometry(X, y)
            centroid_records.append(
                {
                    "model": model,
                    "draw": draw,
                    "layer": layer,
                    "separation_ratio": geom["separation_ratio"],
                }
            )

df_mlp_sep = pd.DataFrame(centroid_records)
save_analysis(df_mlp_sep, "05_mlp_separation_trajectory.csv")
print(f"MLP separation records: {len(df_mlp_sep)}")

MLP separation records: 246


In [6]:
# Plot MLP separation ratio trajectory across models
fig, ax = plt.subplots(figsize=(12, 6))
for model in MODELS:
    model_data = df_mlp_sep[
        (df_mlp_sep["model"] == model) & (df_mlp_sep["draw"] == "draw_1")
    ].sort_values("layer")
    if len(model_data) == 0:
        continue
    ax.plot(
        model_data["layer"],
        model_data["separation_ratio"],
        color=MODEL_COLORS.get(model, "gray"),
        label=model,
        marker="o",
        markersize=4,
    )

ax.set_xlabel("Layer")
ax.set_ylabel("Separation Ratio")
ax.set_title("MLP Output Band Separation Across Layers")
ax.legend()
save_figure(fig, "viz_05_02_mlp_separation_trajectory.png")

## 3. MLP Separability (Linear Probing)

Linear probe on MLP outputs per layer: can MLP representations alone predict
frequency band? Compare with residual stream probing from NB02.

In [7]:
probe_records = []

for model in MODELS:
    n_layers = MODEL_INFO[model]["n_layers"]
    print(f"\n{model} ({n_layers} layers):")

    # Only use draw_1 for probe trajectory to save time
    for draw in ["draw_1"]:
        # Build per-layer activation dict: probe at key layers only for large models
        if n_layers <= 8:
            probe_layers = list(range(n_layers))
        else:
            # Every other layer + first and last
            probe_layers = sorted(
                set([0] + list(range(0, n_layers, 2)) + [n_layers - 1])
            )

        layer_activations = {}
        labels_list = []

        for layer in probe_layers:
            embs = []
            if layer == probe_layers[0]:
                labels_list = []
            for band in BANDS:
                mlp_out = all_mlp_out.get(model, {}).get(draw, {}).get(band)
                if mlp_out is None:
                    continue
                embs.append(mlp_out[:, layer, :])
                if layer == probe_layers[0]:
                    labels_list.extend([band] * mlp_out.shape[0])

            if len(embs) > 0:
                layer_activations[layer] = np.vstack(embs)

        labels = np.array(labels_list)

        if not layer_activations:
            continue

        # Train probes at selected layers
        trajectory = train_probe_trajectory(layer_activations, labels)

        for entry in trajectory:
            probe_records.append(
                {
                    "model": model,
                    "draw": draw,
                    "layer": entry["layer"],
                    "accuracy": entry["accuracy"],
                    "std": entry["std"],
                    "source": "mlp_out",
                }
            )

        peak = max(trajectory, key=lambda x: x["accuracy"])
        print(
            f"  {draw}: peak MLP probe accuracy = {peak['accuracy']:.3f} at layer {peak['layer']}"
        )

df_mlp_probe = pd.DataFrame(probe_records)
save_analysis(df_mlp_probe, "05_mlp_probe_trajectory.csv")
print(f"\nMLP probe trajectory records: {len(df_mlp_probe)}")


pythia-70m (6 layers):


  draw_1: peak MLP probe accuracy = 0.556 at layer 3

pythia-160m (12 layers):


  draw_1: peak MLP probe accuracy = 0.605 at layer 11

pythia-410m (24 layers):


  draw_1: peak MLP probe accuracy = 0.656 at layer 23

pythia-1b (16 layers):


  draw_1: peak MLP probe accuracy = 0.637 at layer 15

pythia-1.4b (24 layers):


  draw_1: peak MLP probe accuracy = 0.676 at layer 23

MLP probe trajectory records: 48


In [8]:
# Load NB02 residual stream probe results for comparison
resid_probe_path = ANALYSIS_DIR / "02_probe_trajectory.csv"
df_resid_probe = None
if resid_probe_path.exists():
    df_resid_probe = pd.read_csv(resid_probe_path)
    df_resid_probe["source"] = "resid_post"
    print(f"Loaded NB02 residual probe data: {len(df_resid_probe)} records")
else:
    print("NB02 probe trajectory not found; skipping comparison")

NB02 probe trajectory not found; skipping comparison


In [9]:
# Plot MLP vs residual stream probe trajectories per model
for model in MODELS:
    fig, ax = plt.subplots(figsize=(12, 6))

    # MLP probe
    mlp_data = df_mlp_probe[
        (df_mlp_probe["model"] == model) & (df_mlp_probe["draw"] == "draw_1")
    ].sort_values("layer")
    if len(mlp_data) > 0:
        ax.plot(
            mlp_data["layer"],
            mlp_data["accuracy"],
            color="#ff7f0e",
            label="MLP output",
            marker="s",
            markersize=4,
        )
        ax.fill_between(
            mlp_data["layer"],
            mlp_data["accuracy"] - mlp_data["std"],
            mlp_data["accuracy"] + mlp_data["std"],
            color="#ff7f0e",
            alpha=0.15,
        )

    # Residual stream probe
    if df_resid_probe is not None:
        resid_data = df_resid_probe[
            (df_resid_probe["model"] == model) & (df_resid_probe["draw"] == "draw_1")
        ].sort_values("layer")
        if len(resid_data) > 0:
            ax.plot(
                resid_data["layer"],
                resid_data["accuracy"],
                color="#1f77b4",
                label="Residual stream",
                marker="o",
                markersize=4,
            )
            if "std" in resid_data.columns:
                ax.fill_between(
                    resid_data["layer"],
                    resid_data["accuracy"] - resid_data["std"],
                    resid_data["accuracy"] + resid_data["std"],
                    color="#1f77b4",
                    alpha=0.15,
                )

    ax.axhline(
        y=1.0 / len(BANDS), color="gray", linestyle="--", alpha=0.5, label="Chance"
    )
    ax.set_xlabel("Layer")
    ax.set_ylabel("Probe Accuracy (5-fold CV)")
    ax.set_title(f"MLP vs Residual Stream Probe Trajectory \u2014 {model}")
    ax.legend()
    ax.set_ylim(bottom=0)
    save_figure(fig, f"viz_05_03_mlp_vs_resid_probe_{model}.png")

## 4. MLP Contribution to Correct Logit

Compute `mlp_out @ W_U[target]` per layer per band, measuring how much each
MLP layer pushes the logit of the correct target token.

**Note**: This requires access to the unembedding matrix W_U. We attempt to
load it from the model. If unavailable, we use MLP output norm as a proxy
for contribution magnitude.

In [10]:
# Attempt to load W_U (unembedding matrix) for logit attribution
# If not available, fall back to norm-based proxy

W_U_loaded = {}  # model -> W_U array (d_model, vocab_size)

try:
    import torch
    from transformers import AutoModelForCausalLM

    HAS_TORCH = True
except ImportError:
    HAS_TORCH = False
    print("torch/transformers not available; using norm-based proxy")

if HAS_TORCH:
    from utils.constants import HF_MODEL_NAMES

    for model in MODELS:
        hf_name = HF_MODEL_NAMES.get(model)
        if hf_name is None:
            continue
        try:
            hf_model = AutoModelForCausalLM.from_pretrained(hf_name)
            # Pythia models: embed_out is the unembedding layer
            if hasattr(hf_model, "embed_out"):
                W_U_loaded[model] = (
                    hf_model.embed_out.weight.detach().cpu().numpy()
                )  # (vocab, d_model)
            elif hasattr(hf_model, "lm_head"):
                W_U_loaded[model] = (
                    hf_model.lm_head.weight.detach().cpu().numpy()
                )  # (vocab, d_model)
            del hf_model
            if HAS_TORCH:
                torch.cuda.empty_cache()
            print(f"  Loaded W_U for {model}: shape {W_U_loaded[model].shape}")
        except Exception as e:
            print(f"  Could not load W_U for {model}: {e}")

USE_LOGIT_LENS = len(W_U_loaded) > 0
print(f"\nLogit attribution mode: {'direct (W_U)' if USE_LOGIT_LENS else 'norm proxy'}")

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

  Loaded W_U for pythia-70m: shape (50304, 512)


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

  Loaded W_U for pythia-160m: shape (50304, 768)


Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

  Loaded W_U for pythia-410m: shape (50304, 1024)


Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

  Loaded W_U for pythia-1b: shape (50304, 2048)


Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

  Loaded W_U for pythia-1.4b: shape (50304, 2048)

Logit attribution mode: direct (W_U)


In [11]:
logit_records = []

for model in MODELS:
    n_layers = MODEL_INFO[model]["n_layers"]
    W_U = W_U_loaded.get(model)  # (vocab, d_model) or None

    for draw in DRAWS:
        for band in BANDS:
            mlp_out = all_mlp_out.get(model, {}).get(draw, {}).get(band)
            targets = all_target_ids.get(model, {}).get(draw, {}).get(band)
            if mlp_out is None:
                continue

            N = mlp_out.shape[0]

            for layer in range(n_layers):
                X = mlp_out[:, layer, :]  # (N, d_model)

                if W_U is not None and targets is not None:
                    # Direct logit attribution: mlp_out_i dot W_U[target_i]
                    target_ids_int = targets.astype(int)
                    W_target = W_U[target_ids_int]  # (N, d_model)
                    logit_contribs = np.sum(X * W_target, axis=1)  # (N,)
                    logit_records.append(
                        {
                            "model": model,
                            "draw": draw,
                            "band": band,
                            "layer": layer,
                            "mean_logit_contrib": float(logit_contribs.mean()),
                            "std_logit_contrib": float(logit_contribs.std()),
                            "median_logit_contrib": float(np.median(logit_contribs)),
                            "method": "direct",
                        }
                    )
                else:
                    # Fallback: use norm as proxy
                    norms = np.linalg.norm(X, axis=1)
                    logit_records.append(
                        {
                            "model": model,
                            "draw": draw,
                            "band": band,
                            "layer": layer,
                            "mean_logit_contrib": float(norms.mean()),
                            "std_logit_contrib": float(norms.std()),
                            "median_logit_contrib": float(np.median(norms)),
                            "method": "norm_proxy",
                        }
                    )

df_logit = pd.DataFrame(logit_records)
save_analysis(df_logit, "05_mlp_logit_contribution.csv")
print(f"Logit contribution records: {len(df_logit)}")
if len(df_logit) > 0:
    print(f"Method: {df_logit['method'].iloc[0]}")

Logit contribution records: 1230
Method: direct


In [12]:
# Plot MLP logit contribution trajectory per model, colored by band
metric_col = "mean_logit_contrib"
method_label = "Direct Logit" if USE_LOGIT_LENS else "Norm Proxy"

for model in MODELS:
    model_data = df_logit[(df_logit["model"] == model) & (df_logit["draw"] == "draw_1")]
    if len(model_data) == 0:
        continue

    fig, ax = plt.subplots(figsize=(12, 5))
    for band in BANDS:
        bd = model_data[model_data["band"] == band].sort_values("layer")
        if len(bd) == 0:
            continue
        ax.plot(
            bd["layer"],
            bd[metric_col],
            color=BAND_COLORS.get(band, "gray"),
            label=BAND_NAMES.get(band, band),
            marker="o",
            markersize=3,
        )
    ax.set_xlabel("Layer")
    ax.set_ylabel(f"MLP Contribution ({method_label})")
    ax.set_title(f"MLP Contribution to Correct Logit \u2014 {model}")
    ax.legend(fontsize=8)
    fig.tight_layout()
    save_figure(fig, f"viz_05_04_mlp_logit_contrib_{model}.png")

In [13]:
# Heatmap: mean MLP logit contribution at the peak layer per model x band
peak_logit_records = []
for model in MODELS:
    model_data = df_logit[(df_logit["model"] == model) & (df_logit["draw"] == "draw_1")]
    if len(model_data) == 0:
        continue
    for band in BANDS:
        bd = model_data[model_data["band"] == band]
        if len(bd) == 0:
            continue
        peak_row = bd.loc[bd[metric_col].abs().idxmax()]
        peak_logit_records.append(
            {
                "model": model,
                "band": band,
                "peak_logit_contrib": peak_row[metric_col],
                "peak_layer": int(peak_row["layer"]),
            }
        )

if peak_logit_records:
    df_peak_logit = pd.DataFrame(peak_logit_records)
    pivot = df_peak_logit.pivot(
        index="model", columns="band", values="peak_logit_contrib"
    )
    pivot = pivot.reindex(index=MODELS, columns=BANDS)
    fig = plot_metric_heatmap(
        pivot,
        f"Peak MLP Logit Contribution ({method_label})",
        fmt=".2f",
        cmap="YlOrRd",
    )
    save_figure(fig, "viz_05_05_peak_logit_heatmap.png")

## 5. MLP Contribution Fraction (vs Attention)

MLP share of total representation update per layer.
The residual stream update at each layer is `attn_out + mlp_out`.
We compute the fraction: `||mlp_out|| / (||mlp_out|| + ||attn_out||)`.

In [14]:
frac_records = []

for model in MODELS:
    n_layers = MODEL_INFO[model]["n_layers"]
    for draw in DRAWS:
        for band in BANDS:
            mlp_out = all_mlp_out.get(model, {}).get(draw, {}).get(band)
            attn_out = all_attn_out.get(model, {}).get(draw, {}).get(band)
            if mlp_out is None or attn_out is None:
                continue

            for layer in range(n_layers):
                mlp_norms = np.linalg.norm(mlp_out[:, layer, :], axis=1)  # (N,)
                attn_norms = np.linalg.norm(attn_out[:, layer, :], axis=1)  # (N,)
                total_norms = mlp_norms + attn_norms
                # Avoid division by zero
                safe_total = np.maximum(total_norms, 1e-10)
                mlp_frac = mlp_norms / safe_total
                attn_frac = attn_norms / safe_total

                frac_records.append(
                    {
                        "model": model,
                        "draw": draw,
                        "band": band,
                        "layer": layer,
                        "mlp_frac_mean": float(mlp_frac.mean()),
                        "mlp_frac_std": float(mlp_frac.std()),
                        "attn_frac_mean": float(attn_frac.mean()),
                        "mlp_norm_mean": float(mlp_norms.mean()),
                        "attn_norm_mean": float(attn_norms.mean()),
                    }
                )

df_frac = pd.DataFrame(frac_records)
save_analysis(df_frac, "05_mlp_contribution_fraction.csv")
print(f"MLP fraction records: {len(df_frac)}")

MLP fraction records: 1230


In [15]:
# Plot MLP fraction trajectory per model, colored by band
for model in MODELS:
    model_data = df_frac[(df_frac["model"] == model) & (df_frac["draw"] == "draw_1")]
    if len(model_data) == 0:
        continue

    fig, ax = plt.subplots(figsize=(12, 5))
    for band in BANDS:
        bd = model_data[model_data["band"] == band].sort_values("layer")
        if len(bd) == 0:
            continue
        ax.plot(
            bd["layer"],
            bd["mlp_frac_mean"],
            color=BAND_COLORS.get(band, "gray"),
            label=BAND_NAMES.get(band, band),
            marker="o",
            markersize=3,
        )

    ax.axhline(y=0.5, color="gray", linestyle="--", alpha=0.4, label="50% (equal)")
    ax.set_xlabel("Layer")
    ax.set_ylabel("MLP Fraction of Update Norm")
    ax.set_title(f"MLP Contribution Fraction \u2014 {model}")
    ax.set_ylim(0, 1)
    ax.legend(fontsize=8)
    fig.tight_layout()
    save_figure(fig, f"viz_05_06_mlp_fraction_{model}.png")

In [16]:
# Stacked area: MLP vs Attention norms per band (single model example)
for model in MODELS:
    model_data = df_frac[(df_frac["model"] == model) & (df_frac["draw"] == "draw_1")]
    if len(model_data) == 0:
        continue

    n_bands = len(BANDS)
    fig, axes = plt.subplots(1, n_bands, figsize=(4 * n_bands, 4), sharey=True)
    if n_bands == 1:
        axes = [axes]

    for ax, band in zip(axes, BANDS):
        bd = model_data[model_data["band"] == band].sort_values("layer")
        if len(bd) == 0:
            continue
        layers = bd["layer"].values
        attn_frac = bd["attn_frac_mean"].values
        mlp_frac = bd["mlp_frac_mean"].values

        ax.stackplot(
            layers,
            attn_frac,
            mlp_frac,
            labels=["Attention", "MLP"],
            colors=["#1f77b4", "#ff7f0e"],
            alpha=0.7,
        )
        ax.set_title(BAND_NAMES.get(band, band), fontsize=9)
        ax.set_xlabel("Layer")
        if ax == axes[0]:
            ax.set_ylabel("Fraction")

    axes[-1].legend(loc="upper right", fontsize=7)
    fig.suptitle(f"Attention vs MLP Contribution \u2014 {model}", y=1.02)
    fig.tight_layout()
    save_figure(fig, f"viz_05_07_stacked_attn_mlp_{model}.png")

## 6. Neuron-Level Analysis (Frequency Selectivity)

From `mlp_pre` activations, identify frequency-selective neurons.
Selectivity index per neuron: one-way ANOVA F-statistic measuring the
ratio of between-band variance to within-band variance.
Top-K most selective neurons are identified per layer.

In [17]:
TOP_K_NEURONS = 50  # Report top-K selective neurons per layer

selectivity_summary = []  # Per-layer summary
top_neuron_records = []  # Top-K neurons per layer

for model in MODELS:
    n_layers = MODEL_INFO[model]["n_layers"]
    d_mlp = MODEL_D_MLP[model]
    print(f"\n{model} (d_mlp={d_mlp}, {n_layers} layers):")

    for draw in ["draw_1"]:  # Single draw for efficiency (neuron analysis is expensive)
        # Collect all mlp_pre activations and labels
        all_pre_by_band = {}
        for band in BANDS:
            mlp_pre = all_mlp_pre.get(model, {}).get(draw, {}).get(band)
            if mlp_pre is not None:
                all_pre_by_band[band] = mlp_pre  # (N, n_layers, d_mlp)

        if len(all_pre_by_band) < 2:
            print("  Skipping: insufficient bands with mlp_pre data")
            continue

        for layer in range(n_layers):
            # Gather per-band neuron activations at this layer
            groups = []
            band_labels = []
            for band in BANDS:
                if band not in all_pre_by_band:
                    continue
                pre = all_pre_by_band[band][:, layer, :]  # (N_band, d_mlp)
                groups.append(pre)
                band_labels.append(band)

            if len(groups) < 2:
                continue

            # Stack all examples
            all_acts = np.vstack(groups)  # (N_total, d_mlp)
            all_labels = np.concatenate(
                [np.full(g.shape[0], i) for i, g in enumerate(groups)]
            )

            # Compute F-statistic for each neuron using one-way ANOVA
            f_stats = np.zeros(d_mlp)
            p_vals = np.ones(d_mlp)

            for neuron_idx in range(d_mlp):
                neuron_acts = all_acts[:, neuron_idx]
                neuron_groups = [
                    neuron_acts[all_labels == i] for i in range(len(groups))
                ]
                # Filter out groups with < 2 samples
                valid_groups = [g for g in neuron_groups if len(g) >= 2]
                if len(valid_groups) >= 2:
                    try:
                        f_stat, p_val = stats.f_oneway(*valid_groups)
                        if np.isfinite(f_stat):
                            f_stats[neuron_idx] = f_stat
                            p_vals[neuron_idx] = p_val
                    except Exception:
                        pass

            # Summary stats for this layer
            n_selective = int(np.sum(p_vals < 0.01))  # Bonferroni could be applied
            n_highly_selective = int(np.sum(f_stats > 10.0))
            selectivity_summary.append(
                {
                    "model": model,
                    "draw": draw,
                    "layer": layer,
                    "mean_f_stat": float(f_stats.mean()),
                    "median_f_stat": float(np.median(f_stats)),
                    "max_f_stat": float(f_stats.max()),
                    "n_selective_p001": n_selective,
                    "n_highly_selective_f10": n_highly_selective,
                    "frac_selective": n_selective / d_mlp,
                }
            )

            # Top-K neurons
            top_k_idx = np.argsort(f_stats)[::-1][:TOP_K_NEURONS]
            for rank, nidx in enumerate(top_k_idx):
                # Determine which band this neuron is most active for
                group_means = [groups[i][:, nidx].mean() for i in range(len(groups))]
                preferred_band_idx = int(np.argmax(group_means))
                top_neuron_records.append(
                    {
                        "model": model,
                        "draw": draw,
                        "layer": layer,
                        "neuron_idx": int(nidx),
                        "rank": rank,
                        "f_stat": float(f_stats[nidx]),
                        "p_val": float(p_vals[nidx]),
                        "preferred_band": band_labels[preferred_band_idx],
                        "preferred_band_mean": float(group_means[preferred_band_idx]),
                    }
                )

        print(f"  Done: {n_layers} layers")

df_selectivity = pd.DataFrame(selectivity_summary)
df_top_neurons = pd.DataFrame(top_neuron_records)
save_analysis(df_selectivity, "05_neuron_selectivity_summary.csv")
save_analysis(df_top_neurons, "05_top_selective_neurons.csv")
print(f"\nSelectivity summary records: {len(df_selectivity)}")
print(f"Top neuron records: {len(df_top_neurons)}")


pythia-70m (d_mlp=2048, 6 layers):


  Done: 6 layers

pythia-160m (d_mlp=3072, 12 layers):


  Done: 12 layers

pythia-410m (d_mlp=4096, 24 layers):


  Done: 24 layers

pythia-1b (d_mlp=8192, 16 layers):


  Done: 16 layers

pythia-1.4b (d_mlp=8192, 24 layers):


  Done: 24 layers

Selectivity summary records: 82
Top neuron records: 4100


In [18]:
# Plot fraction of selective neurons per layer, across models
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: fraction selective (p < 0.01)
ax = axes[0]
for model in MODELS:
    md = df_selectivity[df_selectivity["model"] == model].sort_values("layer")
    if len(md) == 0:
        continue
    ax.plot(
        md["layer"],
        md["frac_selective"],
        color=MODEL_COLORS.get(model, "gray"),
        label=model,
        marker="o",
        markersize=4,
    )
ax.set_xlabel("Layer")
ax.set_ylabel("Fraction Selective (p < 0.01)")
ax.set_title("Frequency-Selective Neurons per Layer")
ax.legend(fontsize=8)

# Right: mean F-statistic
ax = axes[1]
for model in MODELS:
    md = df_selectivity[df_selectivity["model"] == model].sort_values("layer")
    if len(md) == 0:
        continue
    ax.plot(
        md["layer"],
        md["mean_f_stat"],
        color=MODEL_COLORS.get(model, "gray"),
        label=model,
        marker="o",
        markersize=4,
    )
ax.set_xlabel("Layer")
ax.set_ylabel("Mean F-Statistic")
ax.set_title("Mean Neuron Selectivity (F-stat) per Layer")
ax.legend(fontsize=8)

fig.tight_layout()
save_figure(fig, "viz_05_08_neuron_selectivity_trajectory.png")

In [19]:
# For each model, which bands do the top-K neurons prefer?
for model in MODELS:
    model_neurons = df_top_neurons[
        (df_top_neurons["model"] == model) & (df_top_neurons["rank"] < 20)
    ]
    if len(model_neurons) == 0:
        continue

    n_layers = MODEL_INFO[model]["n_layers"]

    # Count preferred band per layer (for top-20 neurons)
    band_counts = np.zeros((n_layers, len(BANDS)))
    band_to_idx = {b: i for i, b in enumerate(BANDS)}

    for _, row in model_neurons.iterrows():
        layer = int(row["layer"])
        band = row["preferred_band"]
        if band in band_to_idx and layer < n_layers:
            band_counts[layer, band_to_idx[band]] += 1

    # Create DataFrame with layers as rows, bands as columns
    df_counts = pd.DataFrame(
        band_counts,
        index=range(n_layers),
        columns=[BAND_NAMES.get(b, b) for b in BANDS],
    )

    # Transpose so bands are on Y-axis (rows) and layers on X-axis (columns)
    fig, ax = plt.subplots(figsize=(max(10, n_layers * 0.5), 6))
    sns.heatmap(
        df_counts.T,
        annot=True,
        fmt=".0f",
        cmap="YlOrRd",
        square=True,
        linewidths=0,
        linecolor="none",
        ax=ax,
    )
    ax.set_xlabel("Layer")
    ax.set_ylabel("Preferred Band")
    ax.set_title(
        f"Top-20 Selective Neurons: Preferred Band Distribution \u2014 {model}"
    )
    save_figure(fig, f"viz_05_09_preferred_band_dist_{model}.png")

## 7. MLP Activation Sparsity

Sparsity of MLP pre-activations per band, measured by Gini coefficient
of activation magnitudes. Higher Gini means a few neurons dominate
(more sparse/specialized representation).

In [20]:
def gini_coefficient(x):
    """Compute Gini coefficient of an array of values.

    Higher values indicate more inequality (sparsity).
    Returns value in [0, 1].
    """
    x = np.abs(x)
    if np.sum(x) == 0:
        return 0.0
    x = np.sort(x)
    n = len(x)
    index = np.arange(1, n + 1)
    return float((2 * np.sum(index * x) / (n * np.sum(x))) - (n + 1) / n)


def fraction_active(x, threshold=0.01):
    """Fraction of neurons with |activation| > threshold * max."""
    x = np.abs(x)
    max_val = x.max()
    if max_val == 0:
        return 0.0
    return float(np.mean(x > threshold * max_val))


sparsity_records = []

for model in MODELS:
    n_layers = MODEL_INFO[model]["n_layers"]
    for draw in DRAWS:
        for band in BANDS:
            mlp_pre = all_mlp_pre.get(model, {}).get(draw, {}).get(band)
            if mlp_pre is None:
                continue

            for layer in range(n_layers):
                X = mlp_pre[:, layer, :]  # (N, d_mlp)
                N = X.shape[0]

                # Compute Gini for each example and average
                gini_vals = [gini_coefficient(X[i]) for i in range(N)]
                frac_active_vals = [fraction_active(X[i]) for i in range(N)]

                # Also compute population-level Gini (across mean activation magnitude per neuron)
                mean_abs_act = np.abs(X).mean(axis=0)  # (d_mlp,)
                pop_gini = gini_coefficient(mean_abs_act)

                sparsity_records.append(
                    {
                        "model": model,
                        "draw": draw,
                        "band": band,
                        "layer": layer,
                        "mean_gini": float(np.mean(gini_vals)),
                        "std_gini": float(np.std(gini_vals)),
                        "population_gini": pop_gini,
                        "mean_frac_active": float(np.mean(frac_active_vals)),
                        "mean_abs_activation": float(np.abs(X).mean()),
                    }
                )

df_sparsity = pd.DataFrame(sparsity_records)
save_analysis(df_sparsity, "05_mlp_sparsity.csv")
print(f"Sparsity records: {len(df_sparsity)}")

Sparsity records: 1230


In [21]:
# Plot Gini coefficient trajectory per model, colored by band
for model in MODELS:
    model_data = df_sparsity[
        (df_sparsity["model"] == model) & (df_sparsity["draw"] == "draw_1")
    ]
    if len(model_data) == 0:
        continue

    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    # Gini coefficient
    ax = axes[0]
    for band in BANDS:
        bd = model_data[model_data["band"] == band].sort_values("layer")
        if len(bd) == 0:
            continue
        ax.plot(
            bd["layer"],
            bd["mean_gini"],
            color=BAND_COLORS.get(band, "gray"),
            label=BAND_NAMES.get(band, band),
            marker="o",
            markersize=3,
        )
    ax.set_xlabel("Layer")
    ax.set_ylabel("Mean Gini Coefficient")
    ax.set_title(f"MLP Activation Sparsity (Gini) \u2014 {model}")
    ax.legend(fontsize=8)

    # Fraction active
    ax = axes[1]
    for band in BANDS:
        bd = model_data[model_data["band"] == band].sort_values("layer")
        if len(bd) == 0:
            continue
        ax.plot(
            bd["layer"],
            bd["mean_frac_active"],
            color=BAND_COLORS.get(band, "gray"),
            label=BAND_NAMES.get(band, band),
            marker="o",
            markersize=3,
        )
    ax.set_xlabel("Layer")
    ax.set_ylabel("Fraction of Neurons Active")
    ax.set_title(f"Fraction Active Neurons \u2014 {model}")
    ax.legend(fontsize=8)

    fig.tight_layout()
    save_figure(fig, f"viz_05_10_sparsity_trajectory_{model}.png")

In [22]:
# Boxplot: Gini coefficient across layers per band (aggregated over layers)
for model in MODELS:
    model_data = df_sparsity[
        (df_sparsity["model"] == model) & (df_sparsity["draw"] == "draw_1")
    ].copy()
    if len(model_data) == 0:
        continue

    fig = plot_boxplot_by_band(
        model_data,
        x="band",
        y="mean_gini",
        title=f"MLP Sparsity Distribution by Band \u2014 {model}",
        ylabel="Mean Gini Coefficient",
    )
    save_figure(fig, f"viz_05_11_sparsity_boxplot_{model}.png")

LSC_circuit_analysis/03_Phase_Representational/utils/plotting.py:97: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(
LSC_circuit_analysis/03_Phase_Representational/utils/plotting.py:102: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.stripplot(


LSC_circuit_analysis/03_Phase_Representational/utils/plotting.py:97: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(
LSC_circuit_analysis/03_Phase_Representational/utils/plotting.py:102: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.stripplot(


LSC_circuit_analysis/03_Phase_Representational/utils/plotting.py:97: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(
LSC_circuit_analysis/03_Phase_Representational/utils/plotting.py:102: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.stripplot(


LSC_circuit_analysis/03_Phase_Representational/utils/plotting.py:97: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(
LSC_circuit_analysis/03_Phase_Representational/utils/plotting.py:102: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.stripplot(


LSC_circuit_analysis/03_Phase_Representational/utils/plotting.py:97: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(
LSC_circuit_analysis/03_Phase_Representational/utils/plotting.py:102: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.stripplot(


## 8. Cross-Model Comparison

Aggregate MLP metrics across model sizes: contribution magnitude,
neuron selectivity, sparsity, and probe accuracy.

In [23]:
master_records = []

for model in MODELS:
    n_layers = MODEL_INFO[model]["n_layers"]
    d_mlp = MODEL_D_MLP[model]

    for draw in DRAWS:
        # Peak MLP probe accuracy
        probe_data = df_mlp_probe[
            (df_mlp_probe["model"] == model) & (df_mlp_probe["draw"] == draw)
        ]
        if len(probe_data) > 0:
            peak_acc = probe_data["accuracy"].max()
            peak_probe_layer = int(
                probe_data.loc[probe_data["accuracy"].idxmax(), "layer"]
            )
        else:
            peak_acc, peak_probe_layer = np.nan, np.nan

        # Mean MLP fraction across layers
        frac_data = df_frac[(df_frac["model"] == model) & (df_frac["draw"] == draw)]
        mean_mlp_frac = (
            frac_data["mlp_frac_mean"].mean() if len(frac_data) > 0 else np.nan
        )

        # Peak separation ratio
        sep_data = df_mlp_sep[
            (df_mlp_sep["model"] == model) & (df_mlp_sep["draw"] == draw)
        ]
        peak_sep = sep_data["separation_ratio"].max() if len(sep_data) > 0 else np.nan

        # Mean sparsity (Gini) across all bands and layers
        sparsity_data = df_sparsity[
            (df_sparsity["model"] == model) & (df_sparsity["draw"] == draw)
        ]
        mean_gini = (
            sparsity_data["mean_gini"].mean() if len(sparsity_data) > 0 else np.nan
        )

        # Selectivity summary (draw_1 only)
        sel_data = df_selectivity[
            (df_selectivity["model"] == model) & (df_selectivity["draw"] == draw)
        ]
        mean_frac_selective = (
            sel_data["frac_selective"].mean() if len(sel_data) > 0 else np.nan
        )
        max_mean_f = sel_data["mean_f_stat"].max() if len(sel_data) > 0 else np.nan

        for band in BANDS:
            # Per-band MLP logit contribution
            logit_band = df_logit[
                (df_logit["model"] == model)
                & (df_logit["draw"] == draw)
                & (df_logit["band"] == band)
            ]
            max_logit = (
                logit_band["mean_logit_contrib"].max()
                if len(logit_band) > 0
                else np.nan
            )

            # Per-band sparsity
            sparsity_band = sparsity_data[sparsity_data["band"] == band]
            band_gini = (
                sparsity_band["mean_gini"].mean() if len(sparsity_band) > 0 else np.nan
            )

            master_records.append(
                {
                    "model": model,
                    "draw": draw,
                    "band": band,
                    "model_capacity": MODEL_CAPACITY[model],
                    "d_mlp": d_mlp,
                    "n_layers": n_layers,
                    "peak_probe_accuracy": peak_acc,
                    "peak_probe_layer": peak_probe_layer,
                    "peak_separation_ratio": peak_sep,
                    "mean_mlp_frac": mean_mlp_frac,
                    "mean_gini": band_gini,
                    "mean_frac_selective": mean_frac_selective,
                    "max_mean_f_stat": max_mean_f,
                    "peak_logit_contrib": max_logit,
                }
            )

df_master = pd.DataFrame(master_records)
save_analysis(df_master, "05_master_mlp_analysis.csv")
print(f"Master MLP analysis records: {len(df_master)}")

# Summary table
summary_cols = [
    "peak_probe_accuracy",
    "peak_separation_ratio",
    "mean_mlp_frac",
    "mean_gini",
    "mean_frac_selective",
]
print("\nCross-model summary (draw_1, averaged over bands):")
print(
    df_master[df_master["draw"] == "draw_1"]
    .groupby("model")[summary_cols]
    .mean()
    .round(3)
)

Master MLP analysis records: 75

Cross-model summary (draw_1, averaged over bands):
             peak_probe_accuracy  peak_separation_ratio  mean_mlp_frac  \
model                                                                    
pythia-1.4b                0.676                 29.787          0.570   
pythia-160m                0.605                 15.547          0.632   
pythia-1b                  0.637                 20.703          0.553   
pythia-410m                0.656                 25.615          0.616   
pythia-70m                 0.556                 14.818          0.570   

             mean_gini  mean_frac_selective  
model                                        
pythia-1.4b      0.368                0.650  
pythia-160m      0.419                0.676  
pythia-1b        0.375                0.624  
pythia-410m      0.392                0.638  
pythia-70m       0.435                0.669  


In [24]:
# Scaling panel: key MLP metrics vs model capacity
draw1_data = df_master[df_master["draw"] == "draw_1"].copy()

if len(draw1_data) > 0:
    fig, axes = plt.subplots(1, 4, figsize=(22, 5))

    metrics = [
        "peak_probe_accuracy",
        "mean_mlp_frac",
        "mean_gini",
        "mean_frac_selective",
    ]
    titles = [
        "Peak Probe Accuracy",
        "Mean MLP Fraction",
        "Mean Gini (Sparsity)",
        "Frac. Selective Neurons",
    ]

    for ax, metric, title in zip(axes, metrics, titles):
        for band in BANDS:
            bd = draw1_data[draw1_data["band"] == band]
            if len(bd) == 0:
                continue
            # Average across draws if multiple
            bd_agg = bd.groupby("model_capacity")[metric].mean().reset_index()
            ax.plot(
                bd_agg["model_capacity"],
                bd_agg[metric],
                color=BAND_COLORS.get(band, "gray"),
                label=BAND_NAMES.get(band, band),
                marker="o",
                markersize=5,
            )
        ax.set_xlabel("Model Capacity (M params)")
        ax.set_title(title)
        ax.set_xscale("log")

    axes[0].legend(fontsize=7, loc="best")
    fig.tight_layout()
    save_figure(fig, "viz_05_12_scaling_panel.png")

In [25]:
# Heatmap: peak MLP probe accuracy (model x draw)
probe_summary = df_mlp_probe.groupby(["model", "draw"])["accuracy"].max().reset_index()
if len(probe_summary) > 0:
    pivot = probe_summary.pivot(index="model", columns="draw", values="accuracy")
    pivot = pivot.reindex(index=MODELS, columns=DRAWS)
    fig = plot_metric_heatmap(
        pivot,
        "Peak MLP Probe Accuracy (per model, per draw)",
        xlabel="Draw",
        ylabel="Model",
        fmt=".3f",
        cmap="YlGn",
    )
    save_figure(fig, "viz_05_13_peak_probe_heatmap.png")

In [26]:
# Heatmap: mean Gini coefficient per model x band (averaged across layers)
sparsity_agg = (
    df_sparsity[df_sparsity["draw"] == "draw_1"]
    .groupby(["model", "band"])["mean_gini"]
    .mean()
    .reset_index()
)
if len(sparsity_agg) > 0:
    pivot = sparsity_agg.pivot(index="model", columns="band", values="mean_gini")
    pivot = pivot.reindex(index=MODELS, columns=BANDS)
    fig = plot_metric_heatmap(
        pivot,
        "Mean MLP Sparsity (Gini) by Model and Band",
        fmt=".3f",
        cmap="Purples",
    )
    save_figure(fig, "viz_05_14_sparsity_heatmap.png")

## 9. Summary

In [27]:
print("=" * 70)
print("NOTEBOOK 05: MLP ANALYSIS COMPLETE")
print("=" * 70)

print("\n--- Key Findings ---")

# 1. MLP probe peak
if len(df_mlp_probe) > 0:
    for model in MODELS:
        md = df_mlp_probe[
            (df_mlp_probe["model"] == model) & (df_mlp_probe["draw"] == "draw_1")
        ]
        if len(md) > 0:
            peak = md.loc[md["accuracy"].idxmax()]
            print(
                f"  {model}: peak MLP probe accuracy = {peak['accuracy']:.3f} at layer {int(peak['layer'])}"
            )

# 2. MLP fraction
if len(df_frac) > 0:
    overall_frac = (
        df_frac[df_frac["draw"] == "draw_1"].groupby("model")["mlp_frac_mean"].mean()
    )
    print("\n  Mean MLP fraction (across bands & layers):")
    for model in MODELS:
        if model in overall_frac.index:
            print(f"    {model}: {overall_frac[model]:.3f}")

# 3. Neuron selectivity
if len(df_selectivity) > 0:
    for model in MODELS:
        md = df_selectivity[df_selectivity["model"] == model]
        if len(md) > 0:
            peak_sel_layer = md.loc[md["frac_selective"].idxmax()]
            print(
                f"  {model}: max frac_selective = {peak_sel_layer['frac_selective']:.3f} at layer {int(peak_sel_layer['layer'])}"
            )

# 4. Sparsity
if len(df_sparsity) > 0:
    overall_gini = (
        df_sparsity[df_sparsity["draw"] == "draw_1"]
        .groupby("model")["mean_gini"]
        .mean()
    )
    print("\n  Mean Gini coefficient (across bands & layers):")
    for model in MODELS:
        if model in overall_gini.index:
            print(f"    {model}: {overall_gini[model]:.3f}")

print(f"\n--- Output Files ---")
print(f"Analysis CSVs in: {ANALYSIS_DIR}")
for f in sorted(ANALYSIS_DIR.glob("05_*")):
    print(f"  {f.name}")
print(f"\nFigures in: {VIZ_DIR}")
for f in sorted(VIZ_DIR.glob("viz_05_*")):
    print(f"  {f.name}")

print("\n" + "=" * 70)

NOTEBOOK 05: MLP ANALYSIS COMPLETE

--- Key Findings ---
  pythia-70m: peak MLP probe accuracy = 0.556 at layer 3
  pythia-160m: peak MLP probe accuracy = 0.605 at layer 11
  pythia-410m: peak MLP probe accuracy = 0.656 at layer 23
  pythia-1b: peak MLP probe accuracy = 0.637 at layer 15
  pythia-1.4b: peak MLP probe accuracy = 0.676 at layer 23

  Mean MLP fraction (across bands & layers):
    pythia-70m: 0.570
    pythia-160m: 0.632
    pythia-410m: 0.616
    pythia-1b: 0.553
    pythia-1.4b: 0.570
  pythia-70m: max frac_selective = 0.839 at layer 0
  pythia-160m: max frac_selective = 0.952 at layer 0
  pythia-410m: max frac_selective = 0.997 at layer 0
  pythia-1b: max frac_selective = 0.986 at layer 0
  pythia-1.4b: max frac_selective = 0.993 at layer 0

  Mean Gini coefficient (across bands & layers):
    pythia-70m: 0.435
    pythia-160m: 0.419
    pythia-410m: 0.392
    pythia-1b: 0.375
    pythia-1.4b: 0.368

--- Output Files ---
Analysis CSVs in: LSC_circuit_analysis/03_Phase_